# Imports

In [25]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [26]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [27]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one.parquet')

In [28]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [29]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


# Machine Learning

In [30]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [31]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.001, 10.0, log=True)
    w1 = trial.suggest_float('weight_class_1', 0.001, 10.0, log=True)
    w2 = trial.suggest_float('weight_class_2', 0.001, 10.0, log=True)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=3000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-03 10:56:23,851] A new study created in memory with name: no-name-d9b36f9c-5c10-4462-b0b8-cb76e6fa4e20
Best trial: 4. Best value: 0.948606:   0%|▍                                                                                                                                      | 11/3000 [00:00<01:30, 33.04it/s]

[I 2026-07-03 10:56:23,989] Trial 2 finished with value: 0.6653171354734482 and parameters: {'weight_class_0': 3.4734616757219596, 'weight_class_1': 3.805816637382215, 'weight_class_2': 0.006173677388354917}. Best is trial 2 with value: 0.6653171354734482.
[I 2026-07-03 10:56:23,990] Trial 0 finished with value: 0.9484115427310996 and parameters: {'weight_class_0': 0.007850442713920384, 'weight_class_1': 0.04462153966419699, 'weight_class_2': 0.091764862578854}. Best is trial 0 with value: 0.9484115427310996.
[I 2026-07-03 10:56:24,012] Trial 6 finished with value: 0.8273813024977928 and parameters: {'weight_class_0': 0.8593937614935693, 'weight_class_1': 0.2919638274421619, 'weight_class_2': 0.037978709952975344}. Best is trial 0 with value: 0.9484115427310996.
[I 2026-07-03 10:56:24,013] Trial 7 finished with value: 0.9202340536455934 and parameters: {'weight_class_0': 0.0015290766017109277, 'weight_class_1': 0.024051289610042364, 'weight_class_2': 0.27459538655705307}. Best is trial

[I 2026-07-03 10:56:24,128] Trial 12 finished with value: 0.5896562075934612 and parameters: {'weight_class_0': 4.699477916065641, 'weight_class_1': 0.0020871173027371677, 'weight_class_2': 2.4344377438591183}. Best is trial 4 with value: 0.948605793512619.
[I 2026-07-03 10:56:24,141] Trial 13 finished with value: 0.7928631331878759 and parameters: {'weight_class_0': 0.09438511067175223, 'weight_class_1': 8.742955064996934, 'weight_class_2': 0.016677974448334534}. Best is trial 4 with value: 0.948605793512619.
[I 2026-07-03 10:56:24,159] Trial 15 finished with value: 0.9277510167373487 and parameters: {'weight_class_0': 0.21650747929846784, 'weight_class_1': 0.3182147368967358, 'weight_class_2': 5.495229189107336}. Best is trial 4 with value: 0.948605793512619.
[I 2026-07-03 10:56:24,162] Trial 14 finished with value: 0.6277953319656032 and parameters: {'weight_class_0': 4.534451800751583, 'weight_class_1': 7.762586509773331, 'weight_class_2': 0.0011783082208558933}. Best is trial 4 wi

Best trial: 22. Best value: 0.949305:   1%|█▌                                                                                                                                    | 36/3000 [00:00<00:54, 54.30it/s]

[I 2026-07-03 10:56:24,361] Trial 25 finished with value: 0.929242109998618 and parameters: {'weight_class_0': 0.0053803626124146655, 'weight_class_1': 0.008338980610354476, 'weight_class_2': 0.1442990775691607}. Best is trial 22 with value: 0.9493049080535063.
[I 2026-07-03 10:56:24,381] Trial 26 finished with value: 0.9226699941616158 and parameters: {'weight_class_0': 0.005516668896020284, 'weight_class_1': 0.006849160641645899, 'weight_class_2': 0.15690555380894392}. Best is trial 22 with value: 0.9493049080535063.
[I 2026-07-03 10:56:24,388] Trial 27 finished with value: 0.9481293916340127 and parameters: {'weight_class_0': 0.004192771774970207, 'weight_class_1': 0.14301856917801253, 'weight_class_2': 0.09210607087451052}. Best is trial 22 with value: 0.9493049080535063.
[I 2026-07-03 10:56:24,418] Trial 29 finished with value: 0.9469339013968282 and parameters: {'weight_class_0': 0.0034578573439702567, 'weight_class_1': 0.11598514946017069, 'weight_class_2': 0.15144311569541186}.

...

Best trial: 1741. Best value: 0.949737: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍| 2988/3000 [01:14<00:00, 27.72it/s]

[I 2026-07-03 10:57:38,278] Trial 2985 finished with value: 0.9492423283283914 and parameters: {'weight_class_0': 0.0030736788663982646, 'weight_class_1': 0.030739998685314122, 'weight_class_2': 0.04417740104798558}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,294] Trial 2984 finished with value: 0.9485917152080945 and parameters: {'weight_class_0': 0.005201126244872793, 'weight_class_1': 0.029636492256027953, 'weight_class_2': 0.044036866120058984}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,308] Trial 2986 finished with value: 0.9485329058678807 and parameters: {'weight_class_0': 0.005245981421158441, 'weight_class_1': 0.029871356357539222, 'weight_class_2': 0.043610888750325226}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,347] Trial 2987 finished with value: 0.9195409875527455 and parameters: {'weight_class_0': 0.00881407434868375, 'weight_class_1': 1.2100953692103085, 'weight_class_2': 0.043

[I 2026-07-03 10:57:38,413] Trial 2988 finished with value: 0.9490812683906661 and parameters: {'weight_class_0': 0.0050943357518784494, 'weight_class_1': 0.03528847312838627, 'weight_class_2': 0.04273092746901227}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,420] Trial 2990 finished with value: 0.9445119350215109 and parameters: {'weight_class_0': 0.008876233296078975, 'weight_class_1': 0.03129890143760954, 'weight_class_2': 0.04726115366755209}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,454] Trial 2991 finished with value: 0.9455012945540512 and parameters: {'weight_class_0': 0.00848184302932191, 'weight_class_1': 0.03421803794514479, 'weight_class_2': 0.04381978460299145}. Best is trial 1741 with value: 0.9497368888684448.


Best trial: 1741. Best value: 0.949737: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [01:14<00:00, 40.13it/s]

[I 2026-07-03 10:57:38,478] Trial 2992 finished with value: 0.9474425477203687 and parameters: {'weight_class_0': 0.005374661861463073, 'weight_class_1': 0.030883730608804796, 'weight_class_2': 0.02874785576559878}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,539] Trial 2993 finished with value: 0.9491811935175042 and parameters: {'weight_class_0': 0.005135790316315338, 'weight_class_1': 0.03702790037856549, 'weight_class_2': 0.050851852439994065}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,547] Trial 2994 finished with value: 0.9459279262591824 and parameters: {'weight_class_0': 0.00856330144922181, 'weight_class_1': 0.034644776677395815, 'weight_class_2': 0.049215454240132975}. Best is trial 1741 with value: 0.9497368888684448.
[I 2026-07-03 10:57:38,583] Trial 2995 finished with value: 0.9450792892675608 and parameters: {'weight_class_0': 0.008324993045272847, 'weight_class_1': 0.02974016030471099, 'weight_class_2': 0.0506

In [32]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, ['xgb_0', 'xgb_1', 'xgb_2']].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [33]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [34]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.0.0_xgboost_submission.csv', index=False)

In [35]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
